In [1]:
# Or Shulrufer's binary ensemble design, adapted for independent calibration and current scikit-learn.
# Original source: https://github.com/OrShulrufer/Ensembles/blob/main/ensemble_classifier.py
import importlib.util
import subprocess
import sys
from importlib.metadata import version
from packaging.version import Version

missing_packages = [package for module, package in [("xgboost", "xgboost"), ("plotly", "plotly")] if importlib.util.find_spec(module) is None]
if importlib.util.find_spec("sklearn") is None or Version(version("scikit-learn")) < Version("1.6"):
    missing_packages.append("scikit-learn>=1.6")
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing_packages])

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.datasets import make_classification
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import accuracy_score, auc, average_precision_score, balanced_accuracy_score, confusion_matrix, f1_score, make_scorer, precision_recall_curve, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.validation import check_is_fitted, check_X_y, check_array
from xgboost import XGBClassifier

pio.templates.default = "plotly_white"
SEED = 42


In [2]:
def ensemble_selection_score(expected, probabilities):
    """Combine ROC AUC, average precision and precision-recall AUC using the original weights."""
    precision, recall, _ = precision_recall_curve(expected, probabilities)
    return 0.3 * roc_auc_score(expected, probabilities) + 0.4 * average_precision_score(expected, probabilities) + 0.3 * auc(recall, precision)


class FittedProbabilityVote(ClassifierMixin, BaseEstimator):
    """Average the probabilities of the already fitted binary members in memory."""

    def __init__(self, estimators):
        self.estimators = estimators
        self.classes_ = np.asarray([0, 1])
        self.n_features_in_ = estimators[0].n_features_in_

    def fit(self, X, y):
        return self

    def predict_proba(self, X):
        probabilities = [estimator.predict_proba(X)[:, np.argsort(estimator.classes_)] for estimator in self.estimators]
        return np.mean(probabilities, axis=0)

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]


class EnsembleClassifier(ClassifierMixin, BaseEstimator):
    """
    Fit a calibrated soft-voting ensemble for an imbalanced binary target.

    Parameters
    ----------
    max_datasets : int, default=100
        Maximum number of balanced training subsets. Every subset contains all
        positive training records and an equally sized, disjoint negative chunk.
        The larger of the two sorted class labels is the positive class.

    min_precision : float or None, default=None
        Minimum precision required when choosing a decision threshold on the
        reserved threshold records. Values lie between zero and one.

    min_recall : float or None, default=None
        Minimum recall required on the same threshold records. Both constraints
        apply when supplied together. Fit raises an error if none is attainable.

    Notes
    -----
    The supplied training records split into 80% fitting, 16% calibration and
    4% threshold selection. Calibration and threshold selection retain the
    observed class proportions. An external test set measures the final result.
    This preserves Or Shulrufer's balanced-subset and coordinate-search design.
    Models remain in memory, calibration uses reserved records, and probability
    voting uses a dedicated fitted estimator instead of modifying library state.
    """

    def __init__(self, max_datasets=100, min_precision=None, min_recall=None):
        self.max_datasets = max_datasets
        self.min_precision = min_precision
        self.min_recall = min_recall

    def _create_balanced_datasets(self):
        positive_rows = np.flatnonzero(self.targets_ == 1)
        negative_rows = np.random.default_rng(self.random_seed_).permutation(np.flatnonzero(self.targets_ == 0))
        chunk_size = len(positive_rows)
        dataset_count = min(self.max_datasets, len(negative_rows) // chunk_size)
        if dataset_count == 0:
            raise ValueError("The positive class must have no more fitting records than the negative class.")

        datasets = []
        for dataset in range(dataset_count):
            negative_chunk = negative_rows[dataset * chunk_size:(dataset + 1) * chunk_size]
            rows = np.r_[positive_rows, negative_chunk]
            rows = np.random.default_rng(self.random_seed_ + dataset).permutation(rows)
            datasets.append((self.samples_[rows], self.targets_[rows]))

        self.negative_records_used_ = dataset_count * chunk_size
        self.negative_records_unused_ = len(negative_rows) - self.negative_records_used_
        self.balanced_datasets_ = datasets
        return datasets

    def _optimize_classifier(self, clf, X, y):
        # Change one parameter at a time and repeat only when the validation score improves.
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=self.random_seed_)
        scorer = make_scorer(ensemble_selection_score, response_method="predict_proba")
        best_score = cross_val_score(clone(clf), X, y, cv=cv, scoring=scorer).mean()
        improved = True
        while improved:
            improved = False
            for parameter, values in self.param_grids_[clf.__class__.__name__].items():
                search = GridSearchCV(clone(clf), {parameter: values}, cv=cv, scoring=scorer, n_jobs=1, error_score="raise")
                search.fit(X, y)
                if search.best_score_ > best_score + 1e-12:
                    clf = search.best_estimator_
                    best_score = search.best_score_
                    improved = True

        return clone(clf).fit(X, y)

    def create_voting_classifier(self):
        self.voting_clf_ = FittedProbabilityVote([estimator for _, estimator in self.models_])
        return self.voting_clf_

    def create_calibrated_voting_classifier(self):
        # Calibrate the fitted vote on independent records at the observed class prevalence.
        self.create_voting_classifier()
        self.cal_voting_clf_ = CalibratedClassifierCV(FrozenEstimator(self.voting_clf_), method="sigmoid")
        self.cal_voting_clf_.fit(self.cal_samples_, self.cal_targets_)

        # Choose a threshold on a second reserved set, applying both requested constraints.
        probabilities = self.cal_voting_clf_.predict_proba(self.bt_samples_)[:, 1]
        precision, recall, thresholds = precision_recall_curve(self.bt_targets_, probabilities)
        precision, recall = precision[:-1], recall[:-1]
        eligible = np.ones(len(thresholds), dtype=bool)
        if self.min_precision is not None:
            eligible &= precision >= self.min_precision
        if self.min_recall is not None:
            eligible &= recall >= self.min_recall
        if not eligible.any():
            raise ValueError("No threshold satisfies the requested precision and recall on the reserved records.")

        f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(precision), where=(precision + recall) > 0)
        candidates = np.flatnonzero(eligible)
        best = candidates[np.flatnonzero(f1[candidates] == f1[candidates].max())[-1]]
        self.best_threshold_ = float(thresholds[best])
        return self.cal_voting_clf_

    def fit(self, X, y, classifiers):
        if not isinstance(self.max_datasets, int) or self.max_datasets < 1:
            raise ValueError("max_datasets must be a positive integer.")
        for constraint in [self.min_precision, self.min_recall]:
            if constraint is not None and not 0 <= constraint <= 1:
                raise ValueError("Precision and recall constraints must lie between zero and one.")

        X, y = check_X_y(X, y, accept_sparse=False)
        self.label_encoder_ = LabelEncoder().fit(y)
        if len(self.label_encoder_.classes_) != 2:
            raise ValueError("EnsembleClassifier requires exactly two target classes.")
        self.classes_ = self.label_encoder_.classes_
        self.n_features_in_ = X.shape[1]
        self.random_seed_ = 42
        for attribute in ["best_threshold_", "cal_voting_clf_", "voting_clf_"]:
            self.__dict__.pop(attribute, None)
        encoded = self.label_encoder_.transform(y)
        self.samples_, reserved, self.targets_, reserved_targets = train_test_split(X, encoded, test_size=0.2, stratify=encoded, random_state=self.random_seed_)
        self.cal_samples_, self.bt_samples_, self.cal_targets_, self.bt_targets_ = train_test_split(reserved, reserved_targets, test_size=0.2, stratify=reserved_targets, random_state=self.random_seed_)
        if np.bincount(self.targets_, minlength=2).min() < 3:
            raise ValueError("The fitting records need at least three examples of each class for three-fold tuning.")

        self.classifiers_map = {
            "RandomForestClassifier": RandomForestClassifier(n_estimators=100, max_depth=6, min_samples_leaf=6, min_samples_split=8, random_state=self.random_seed_, n_jobs=1),
            "ExtraTreesClassifier": ExtraTreesClassifier(n_estimators=100, max_depth=6, min_samples_leaf=6, min_samples_split=8, bootstrap=True, random_state=self.random_seed_, n_jobs=1),
            "XGBClassifier": XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.3, colsample_bytree=0.6, colsample_bylevel=0.6, colsample_bynode=0.6, reg_lambda=0, eval_metric="logloss", tree_method="hist", random_state=self.random_seed_, n_jobs=1),
        }
        self.param_grids_ = {
            "RandomForestClassifier": {"max_depth": [3, 6], "min_samples_leaf": [3, 6]},
            "ExtraTreesClassifier": {"max_depth": [3, 6], "min_samples_leaf": [3, 6]},
            "XGBClassifier": {"max_depth": [3, 6], "learning_rate": [0.1, 0.3]},
        }
        if not classifiers or any(name not in self.classifiers_map for name in classifiers):
            raise ValueError("Choose RandomForestClassifier, ExtraTreesClassifier or XGBClassifier.")

        self.models_ = []
        for dataset, (samples, targets) in enumerate(self._create_balanced_datasets(), start=1):
            for family in classifiers:
                estimator = self._optimize_classifier(clone(self.classifiers_map[family]), samples, targets)
                self.models_.append((f"{family}_{dataset}", estimator))
                print(f"Fitted {family} on balanced subset {dataset}/{len(self.balanced_datasets_)}")

        self.create_calibrated_voting_classifier()
        self.partition_summary_ = pd.DataFrame([{"Partition": label, "Records": len(targets), "Positive records": int(targets.sum())} for label, targets in [("Fitting pool", self.targets_), ("Calibration", self.cal_targets_), ("Threshold selection", self.bt_targets_)]])
        return self

    def predict_proba(self, X):
        check_is_fitted(self, "best_threshold_")
        X = check_array(X, accept_sparse=False)
        if X.shape[1] != self.n_features_in_:
            raise ValueError("The number of input features differs from the fitting data.")
        return self.cal_voting_clf_.predict_proba(X)

    def predict(self, X):
        encoded = (self.predict_proba(X)[:, 1] >= self.best_threshold_).astype(int)
        return self.label_encoder_.inverse_transform(encoded)


In [3]:
# Keep a final test set outside fitting, calibration and threshold selection.
samples, targets = make_classification(n_samples=10000, n_features=20, n_informative=8, n_redundant=4, weights=[0.98, 0.02], class_sep=1.2, flip_y=0.003, random_state=SEED)
fit_samples, test_samples, fit_targets, test_targets = train_test_split(samples, targets, test_size=0.2, stratify=targets, random_state=SEED)

ensemble = EnsembleClassifier(max_datasets=3)
ensemble.fit(fit_samples, fit_targets, classifiers=["RandomForestClassifier", "ExtraTreesClassifier", "XGBClassifier"])
test_probabilities = ensemble.predict_proba(test_samples)[:, 1]
test_predictions = ensemble.predict(test_samples)

display(ensemble.partition_summary_)
print(f"Balanced subsets: {len(ensemble.balanced_datasets_)} | Fitted members: {len(ensemble.models_)}")
print(f"Negative fitting records used: {ensemble.negative_records_used_} | Unselected negatives: {ensemble.negative_records_unused_}")
print(f"Threshold selected on reserved records: {ensemble.best_threshold_:.4f}")
print(f"External test: {len(test_targets)} records, including {int(test_targets.sum())} positives.")

test_results = pd.DataFrame([{"Decision threshold": threshold, "Accuracy": accuracy_score(test_targets, predicted), "Balanced accuracy": balanced_accuracy_score(test_targets, predicted), "Precision": precision_score(test_targets, predicted, zero_division=0), "Recall": recall_score(test_targets, predicted), "F1": f1_score(test_targets, predicted), "Average precision": average_precision_score(test_targets, test_probabilities), "ROC AUC": roc_auc_score(test_targets, test_probabilities)} for threshold, predicted in [(0.5, (test_probabilities >= 0.5).astype(int)), (ensemble.best_threshold_, test_predictions)]])
display(test_results)


Fitted RandomForestClassifier on balanced subset 1/3
Fitted ExtraTreesClassifier on balanced subset 1/3
Fitted XGBClassifier on balanced subset 1/3
Fitted RandomForestClassifier on balanced subset 2/3
Fitted ExtraTreesClassifier on balanced subset 2/3
Fitted XGBClassifier on balanced subset 2/3
Fitted RandomForestClassifier on balanced subset 3/3
Fitted ExtraTreesClassifier on balanced subset 3/3
Fitted XGBClassifier on balanced subset 3/3
Balanced subsets: 3 | Fitted members: 9
Negative fitting records used: 399 | Unselected negatives: 5868
Threshold selected on reserved records: 0.3184
External test: 2000 records, including 42 positives.


,Partition,Records,Positive records
0,Fitting pool,6400,133
1,Calibration,1280,26
2,Threshold selection,320,7


,Decision threshold,Accuracy,Balanced accuracy,Precision,Recall,F1,Average precision,ROC AUC
0,0.500000,0.9790,0.500000,0.0,0.000000,0.000000,0.314903,0.904057
1,0.318367,0.9815,0.559524,1.0,0.119048,0.212766,0.314903,0.904057


In [4]:
# Show the final test performance at the threshold chosen before this test.
precision, recall, _ = precision_recall_curve(test_targets, test_probabilities)
observed_fraction, mean_probability = calibration_curve(test_targets, test_probabilities, n_bins=8, strategy="quantile")
figure = make_subplots(rows=1, cols=2, subplot_titles=["External test precision and recall", "External test probability calibration"])
figure.add_trace(go.Scatter(x=recall, y=precision, mode="lines", name="Ensemble", line_color="#234737"), row=1, col=1)
figure.add_trace(go.Scatter(x=[recall_score(test_targets, test_predictions)], y=[precision_score(test_targets, test_predictions, zero_division=0)], mode="markers", marker={"size": 12, "color": "#946E37"}, name="Selected threshold"), row=1, col=1)
figure.add_trace(go.Scatter(x=[0, 1], y=[test_targets.mean(), test_targets.mean()], mode="lines", line={"dash": "dot", "color": "#59695F"}, name="Positive prevalence"), row=1, col=1)
figure.add_trace(go.Scatter(x=mean_probability, y=observed_fraction, mode="lines+markers", line_color="#234737", name="Observed frequency", showlegend=False), row=1, col=2)
figure.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line={"dash": "dot", "color": "#59695F"}, name="Perfect calibration"), row=1, col=2)
figure.update_xaxes(range=[0, 1], title_text="Recall", row=1, col=1)
figure.update_yaxes(range=[0, 1], title_text="Precision", row=1, col=1)
figure.update_xaxes(range=[0, 1], title_text="Mean predicted probability", row=1, col=2)
figure.update_yaxes(range=[0, 1], title_text="Observed positive fraction", row=1, col=2)
figure.update_layout(title="Or Shulrufer's binary ensemble", height=500, legend={"orientation": "h", "y": -0.25})
figure.show()

confusion = pd.DataFrame(confusion_matrix(test_targets, test_predictions), index=["Actual negative", "Actual positive"], columns=["Predicted negative", "Predicted positive"])
display(confusion)


,Predicted negative,Predicted positive
Actual negative,1958,0
Actual positive,37,5
